# Debugging MERGE_pipeline.py 
Break down the script step by step to debug 

In [6]:
from datetime import datetime, timedelta, timezone
import time
import inspect
from typing import Dict

import pandas as pd
import xarray as xr
import logging

from merge_log_config import setup_logger, upload_log_to_s3
from merge_hourly_standardization import merge_hourly_standardization
from merge_derive_missing import merge_derive_missing_vars
from merge_clean_vars import merge_reorder_vars, merge_drop_vars
from merge_eraqc_counts import (
    eraqc_counts_native_timestep,
    eraqc_counts_hourly_timestep,
)

from MERGE_pipeline import read_station_metadata, validate_station, read_zarr_dataset, get_var_attrs, convert_xr_to_df, convert_df_to_xr, write_zarr_to_s3

In [11]:
station = "ASOSAWOS_69007093217"
verbose = True

# station1 = "ASOSAWOS_72493023230"
# station2 = "ASOSAWOS_69007093217"

In [9]:
"""
merge_eraqc_counts.py

Generate and exports CSVs with counts of unique QAQC flag values per variable, in native
and hourly timesteps. These counts are used to produce QAQC flag statistics for the QAQC success report.

Functions
---------
- eraqc_counts_native_timestep: QAQC flag counts in the original timestep
- eraqc_counts_hourly_timestep: QAQC flag counts in the hourly (post standardization) timestep

Intended Use
------------
Import into merge workflows to generate information for the QAQC success report.
"""

import pandas as pd
import logging
import inspect


# -----------------------------------------------------------------------------
def eraqc_counts_native_timestep(
    df: pd.DataFrame, network: str, station: str, logger: logging.Logger
) -> None:
    """
    Generates a dataframe of raw qaqc flag value counts for every variable,
    in their native timestep, before hourly standardization.
    Exports the dataframe as a csv to AWS.

    Parameters
    ----------
    df : pd.DataFrame
        station dataset converted to dataframe through QAQC pipeline
    network: str
        network name
    station: str
        station name
    logger : logging.Logger
        Logger instance for recording messages during processing.

    Returns
    -------
    None
    """
    logger.info(f"{inspect.currentframe().f_code.co_name}: Starting...")

    try:
        # identify _eraqc variables
        eraqc_vars = [var for var in df.columns if "_eraqc" in var]

        # filter df for only qaqc columns
        # also replace Nan values with 'no_flag' for two reasons:
        #   1. to enable us to count total observations for the success report
        #   2. to clarify what the Nan value indicates
        df = df[eraqc_vars].fillna("no_flag")

        # generate df of counts of each unique flag for each variable
        # fill all Nan values with 0, since Nan = no observations counted
        flag_counts = df.apply(pd.Series.value_counts).fillna(0)

        # rename columns
        flag_counts.columns = flag_counts.columns.str.replace("_eraqc", "", regex=True)

        # rename index (i.e. eraqc values) and then reset index
        flag_counts = flag_counts.rename_axis("eraqc_flag_values")

        # add row with total observation count
        total_obs_count = len(df)
        flag_counts.loc["total_obs_count"] = [total_obs_count] * flag_counts.shape[1]

        # set all counts to integers, for readability
        flag_counts = flag_counts.astype(int)

        # send file to AWS
        csv_s3_filepath = f"s3://wecc-historical-wx/4_merge_wx/{network}/eraqc_counts_native_timestep/{station}_flag_counts_native_timestep.csv"
        flag_counts.to_csv(csv_s3_filepath, index=True)

        # Update logger
        logger.info(f"Uploaded file to: {csv_s3_filepath}")
        logger.info(f"{inspect.currentframe().f_code.co_name}: Completed successfully")

    except Exception as e:
        logger.error(f"{inspect.currentframe().f_code.co_name}: Failed")
        raise e


# -----------------------------------------------------------------------------
def eraqc_counts_hourly_timestep(
    df: pd.DataFrame, network: str, station: str, logger: logging.Logger
) -> None:
    """
    Generates a dataframe of raw qaqc flag value counts for every variable, for the hourly
    timestep, after hourly standardization. Includes the total observation count.
    Exports the dataframe as a CSV to AWS.

    Parameters
    ----------
    df : pd.DataFrame
        station dataset converted to dataframe through QAQC pipeline
    network: str
        network name
    station: str
        station name
    logger : logging.Logger
        Logger instance for recording messages during processing.

    Returns
    -------
    None
    """
    logger.info(f"{inspect.currentframe().f_code.co_name}: Starting...")

    try:
        # filter out rows that were infilled during hourly standardization
        df = df[df["standardized_infill"] == "n"]

        # identify _eraqc variables
        eraqc_vars = [var for var in df.columns if "_eraqc" in var]

        # filter df for only qaqc columns
        # also replace Nan values with 'no_flag' for two reasons:
        #   1. to enable us to count total observations for the success report
        #   2. to clarify what the Nan value indicates
        df_qaqc = df[eraqc_vars]

        # generate df of counts of each unique flag for each variable
        # fill all Nan values with 0, since Nan = no observations counted
        flag_counts = df_qaqc.apply(
            lambda x: x.str.split(",", expand=True).stack().value_counts()
        ).fillna(0)

        # rename columns
        flag_counts.columns = flag_counts.columns.str.replace("_eraqc", "", regex=True)

        # rename index (i.e. eraqc values) and then reset index
        flag_counts = flag_counts.rename_axis("eraqc_flag_values")

        # replace 'nan' (a string) with 'no_flag', for clarity
        flag_counts = flag_counts.rename(index={"nan": "no_flag"})

        # add row with total observation count
        total_obs_count = len(df)
        flag_counts.loc["total_obs_count"] = [total_obs_count] * flag_counts.shape[1]

        # set all counts to integers, for readability
        flag_counts = flag_counts.astype(int)

        # send file to AWS
        csv_s3_filepath = f"s3://wecc-historical-wx/4_merge_wx/{network}/eraqc_counts_hourly_timestep/{station}_flag_counts_hourly_standardized.csv"
        flag_counts.to_csv(csv_s3_filepath, index=True)

        # Update logger
        logger.info(f"Uploaded file to: {csv_s3_filepath}")
        logger.info(f"{inspect.currentframe().f_code.co_name}: Completed successfully")

    except Exception as e:
        logger.error(f"{inspect.currentframe().f_code.co_name}: Failed")
        raise e

In [12]:
bucket_name = "wecc-historical-wx"
stations_csv_path = f"s3://{bucket_name}/2_clean_wx/temp_clean_all_station_list.csv"
qaqc_dir = "3_qaqc_wx"
merge_dir = "4_merge_wx"

# Log start time
start_time = time.time()

## ======== SETUP ========

# Set up logger
logger, log_filepath = setup_logger(station, verbose=verbose)

# Load station metadata
stations_df = read_station_metadata(stations_csv_path, logger)

# Validate station and get network name
network_name = validate_station(station, stations_df, logger)

## ======== READ IN AND REFORMAT DATA ========

# Load Zarr dataset from S3
ds = read_zarr_dataset(bucket_name, qaqc_dir, network_name, station, logger)

# Get variable attributes from dataset
var_attrs = get_var_attrs(ds, network_name, logger)

# Convert dataset to DataFrame
df = convert_xr_to_df(ds, logger)

# ======== MERGE FUNCTIONS ========

# Part 1: Construct and export table of raw QAQC counts per variable
# For success report
eraqc_counts_native_timestep(df, network_name, station, logger)
df0 = df.copy()
# Part 2: Derive any missing variables
df, var_attrs = merge_derive_missing_vars(df, var_attrs, logger)

df1 = df.copy()

# Part 3: Standardize sub-hourly observations to hourly
df, var_attrs = merge_hourly_standardization(df, var_attrs, logger)

# Part 3b: Construct and export table of raw QAQC counts per variable post-hourly standardization
eraqc_counts_hourly_timestep(df, network_name, station, logger)

# # Part 4: Drops raw _qc variables (DECISION TO MAKE) or provide code to filter
# df2 = df.copy()
# df, var_attrs = merge_drop_vars(df, var_attrs, logger)

# # Part 5: Re-orders variables into final preferred order
# df = merge_reorder_vars(df, logger)

# # ======== CLEANUP & UPLOAD DATA TO S3 ========

# # Convert the cleaned DataFrame to an xarray.Dataset and assign global + variable-level metadata
# ds_merged = convert_df_to_xr(df, ds.attrs, var_attrs, logger)

# # Write the xarray Dataset as a Zarr file to the specified S3 path
# write_zarr_to_s3(
#     ds_merged, bucket_name, merge_dir, network_name, station, logger
# )

2025-06-09 12:16:37,448 - INFO - Starting merge script for station: ASOSAWOS_69007093217
2025-06-09 12:16:40,815 - INFO - get_var_attrs: Set 'pr' units to 'mm' for ASOSAWOS network to resolve error in units attribute.
2025-06-09 12:16:40,816 - INFO - convert_xr_to_df: Starting...
2025-06-09 12:16:43,357 - INFO - convert_xr_to_df: Completed successfully
2025-06-09 12:16:43,431 - INFO - eraqc_counts_native_timestep: Starting...
2025-06-09 12:16:43,603 - INFO - Uploaded file to: s3://wecc-historical-wx/4_merge_wx/ASOSAWOS/eraqc_counts_native_timestep/ASOSAWOS_69007093217_flag_counts_native_timestep.csv
2025-06-09 12:16:43,605 - INFO - eraqc_counts_native_timestep: Completed successfully
2025-06-09 12:16:43,630 - INFO - merge_derive_missing_vars: Starting...
2025-06-09 12:16:43,631 - INFO - tdps_derived is present in station, no derivation necessary.
2025-06-09 12:16:43,632 - INFO - Calculating hurs_derived...
2025-06-09 12:16:43,638 - INFO - Successfully calculated hurs_derived
2025-06-09

In [ ]:
ds_merged

In [ ]:
print(df0.columns)
print(df1.columns)
print(df2.columns)

In [ ]:
var_attrs